In [19]:
from src.graph import * 
from src.align import * 
from utils import * 
import networkx as nx
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
# Trying out assembly graph-based approach to resolving divergent strains. 

# (1) Read recruitment to generate a subset of reads to work from. 
# Start with a seed contig and run bbmap against a sample. 

In [21]:
align_df = align_load('../data/test/alignments.tsv')
reads_df = pd.read_csv('../data/test/reads.csv', index_col=0)
read_ids = reads_df.read_id

print('Num. mapped reads:', len(reads_df))
print('Num. aligned reads:', len(set(align_df['query'].unique().tolist() + align_df['target'].unique().tolist())))
print('Num. iterations:', reads_df.iteration.nunique())

Num. mapped reads: 1081
Num. aligned reads: 1082
Num. iterations: 2


In [22]:
reads_df

,read_id,flag,ref_id,position,mapping_quality,cigar,mate_ref_id,mate_position,template_length,seq,...,mate_reverse_strand,read_1,read_2,secondary,qc_fail,duplicate,supplementary,orientation,read_number,iteration
0,HISEQ08:237:C17PBACXX:3:1207:4027:197198,163,scaffold_1334,1,44,119=,=,246,395,CTTCACAGACGCCGTCCTCACAGGCGCCGACCTTACAGACGCCGAC...,...,True,False,True,False,False,False,False,FR,1,0
1,HISEQ08:237:C17PBACXX:3:1103:6776:51110,99,scaffold_1334,3,44,61=1X88=,=,133,280,TCACAGACGCCGTCCTCACAGGCGCCGACCTTACAGACGCCGACCT...,...,True,True,False,False,False,False,False,FR,0,0
2,HISEQ08:237:C17PBACXX:3:1204:10126:58654,101,scaffold_1334,4,0,*,=,4,0,TCAGACACGTCAACCTCACAGACGCCGTCCTCACAGGCGCCGTCCT...,...,True,True,False,False,False,False,False,XR,0,0
3,HISEQ08:237:C17PBACXX:3:1204:10126:58654,153,scaffold_1334,4,43,77=,=,4,0,CACAGACGCCGTCCTCACAGGCGCCGACCTTACAGACGCCGACCTT...,...,False,False,True,False,False,False,False,RX,1,0
4,HISEQ08:237:C17PBACXX:3:2307:1422:82035,69,scaffold_1334,20,0,*,=,20,0,TCAAATATAAGATTACACGGATATAATAATTCAAAGCGATATGGCT...,...,False,True,False,False,False,False,False,XF,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
176,HISEQ08:237:C17PBACXX:3:1108:1933:53112,337,HISEQ08:237:C17PBACXX:3:1305:9439:120049/1,14,3,114=1X12=,HISEQ08:237:C17PBACXX:3:1204:3258:60851/2,8,0,*,...,False,True,False,True,False,False,False,RF,0,1
177,HISEQ08:237:C17PBACXX:3:1108:1933:53112,417,HISEQ08:237:C17PBACXX:3:1305:9439:120049/1,59,3,71=,HISEQ08:237:C17PBACXX:3:1204:3258:60851/2,17,0,*,...,True,False,True,True,False,False,False,FR,1,1
178,HISEQ08:237:C17PBACXX:3:1101:6585:81548,73,HISEQ08:237:C17PBACXX:3:1101:6585:81548/1,1,45,150=,=,1,0,CCGTATCTTAATTCTTCATCCTGCTTGCATAGTTCATCATGACTGC...,...,False,True,False,False,False,False,False,FX,0,1
179,HISEQ08:237:C17PBACXX:3:1101:6585:81548,133,HISEQ08:237:C17PBACXX:3:1101:6585:81548/1,1,0,*,=,1,0,TCCCCAGGATCCCCAGGATCCCCAGGACCCCGGGACCCCAGACAAA...,...,False,False,True,False,False,False,False,XF,1,1


In [23]:

graph = StringGraph(read_ids=read_ids, align_df=align_df)

# fig, ax = plt.subplots()
# nx.draw(graph.G, ax=ax, node_size=1)

StringGraph.__init__: Initialized a graph with 6486 edges.
StringGraph.__init__: 0 start nodes.
StringGraph.__init__: 0 end nodes.


In [24]:
graph._get_pair_map()

{'365.1.R': ['365.0.F'],
 '114.0.R': ['114.1.F'],
 '353.0.R': ['353.1.F'],
 '337.1.F': ['337.0.R'],
 '434.1.F': ['434.0.R'],
 '226.0.R': ['226.1.F'],
 '434.0.R': ['434.1.F'],
 '210.0.R': ['210.1.F'],
 '61.0.R': ['61.1.F'],
 '186.0.F': ['186.1.R'],
 '96.1.F': ['96.0.R'],
 '131.0.F': ['131.1.R'],
 '363.1.R': ['363.0.F'],
 '306.1.R': ['306.0.F'],
 '478.1.F': ['478.0.R', '478.0.F'],
 '271.1.R': ['271.0.F'],
 '205.0.R': ['205.1.F'],
 '81.0.R': ['81.1.F'],
 '318.1.R': ['318.0.F'],
 '198.0.R': ['198.1.F'],
 '232.0.R': ['232.1.F'],
 '375.1.F': ['375.0.R'],
 '451.1.F': ['451.0.R', '451.0.F'],
 '104.1.F': ['104.0.R'],
 '247.1.F': ['247.0.R'],
 '375.0.R': ['375.1.F'],
 '380.0.R': ['380.1.F'],
 '213.1.R': ['213.0.F'],
 '83.1.F': ['83.0.R'],
 '366.0.F': ['366.1.R'],
 '213.0.F': ['213.1.R'],
 '224.0.F': ['224.1.R'],
 '43.0.F': ['43.1.R', '43.1.F'],
 '291.1.F': ['291.0.R'],
 '96.0.R': ['96.1.F'],
 '51.1.R': ['51.0.F'],
 '478.0.R': ['478.1.F', '478.1.R'],
 '41.0.R': ['41.1.F'],
 '318.0.F': ['318.1.R']